In [1]:
import TankSim as ts
import TankSim_kijun as tskijun
import TankSim_injee as tsinjee

✅ YOLO 모델 로드 완료
📦 Model : models\best.yolov11s.pt
🖥 Device : cpu
🎯 Classes : {0: 'Car', 1: 'House', 2: 'Human1', 3: 'Human2', 4: 'Human3', 5: 'Mine', 6: 'Rock', 7: 'Tank1', 8: 'Tank2', 9: 'Tent', 10: 'Tree', 11: 'Wall'}


In [ ]:
app = ts.Flask(__name__)
@app.route('/detect', methods=['POST'])
def detect():
    tsinjee.detect()
    
    filtered_results = []
    return ts.jsonify(filtered_results)

@app.route('/stereo_image', methods=['POST'])
def stereo_image():        # 오브젝트 좌표, 위협도, 거리 계산은 다 여기서 실시. 
    tskijun.stereo_image()
    
    return ts.jsonify({"result": "success"})
    
@app.route('/info', methods=['POST'])
def info():              # 내 위치값, 회전값등을 가져와야하기 때문에 여기서 LATEST_INFO에 로그데이터를 저장.
    # info는 Log Mode를 켜야만 작동이 되는 함수.
    tskijun.info()
    
    data = ts.request.get_json(force=True)
    if not data:
        return ts.jsonify({"error": "No JSON received"}), 400

    return ts.jsonify({"status": "success", "control": ""})

@app.route('/get_action', methods=['POST'])
def get_action():
    data = ts.request.get_json(force=True)

    position = data.get("position", {})
    turret = data.get("turret", {})

    pos_x = position.get("x", 0)
    pos_y = position.get("y", 0)
    pos_z = position.get("z", 0)

    turret_x = turret.get("x", 0)
    turret_y = turret.get("y", 0)

    print(f"📨 Position received: x={pos_x}, y={pos_y}, z={pos_z}")
    print(f"🎯 Turret received: x={turret_x}, y={turret_y}")

    if combined_commands:
        command = combined_commands.pop(0)
    else:
        command = {
            "moveWS": {"command": "STOP", "weight": 1.0},
            "moveAD": {"command": "", "weight": 0.0},
            "turretQE": {"command": "", "weight": 0.0},
            "turretRF": {"command": "", "weight": 0.0},
            "fire": False
        }

    print("🔁 Sent Combined Action:", command)
    return ts.jsonify(command)

@app.route('/update_bullet', methods=['POST'])
def update_bullet():
    data = ts.request.get_json()
    if not data:
        return ts.jsonify({"status": "ERROR", "message": "Invalid request data"}), 400

    print(f"💥 Bullet Impact at X={data.get('x')}, Y={data.get('y')}, Z={data.get('z')}, Target={data.get('hit')}")
    return ts.jsonify({"status": "OK", "message": "Bullet impact data received"})


@app.route('/set_destination', methods=['POST'])
def set_destination():
    data = ts.request.get_json()
    if not data or "destination" not in data:
        return ts.jsonify({"status": "ERROR", "message": "Missing destination data"}), 400

    try:
        x, y, z = map(float, data["destination"].split(","))
        print(f"🎯 Destination set to: x={x}, y={y}, z={z}")
        return ts.jsonify({"status": "OK", "destination": {"x": x, "y": y, "z": z}})
    except Exception as e:
        return ts.jsonify({"status": "ERROR", "message": f"Invalid format: {str(e)}"}), 400


@app.route('/update_obstacle', methods=['POST'])
def update_obstacle():
    data = ts.request.get_json()
    if not data:
        return ts.jsonify({'status': 'error', 'message': 'No data received'}), 400
    
    print("🪨 Obstacle Data:", data)
    return ts.jsonify({'status': 'success', 'message': 'Obstacle data received'})


@app.route('/collision', methods=['POST'])
def collision():
    data = ts.request.get_json()
    if not data:
        return ts.jsonify({'status': 'error', 'message': 'No collision data received'}), 400

    object_name = data.get('objectName')
    position = data.get('position', {})
    x = position.get('x')
    y = position.get('y')
    z = position.get('z')

    print(f"💥 Collision Detected - Object: {object_name}, Position: ({x}, {y}, {z})")

    return ts.jsonify({'status': 'success', 'message': 'Collision data received'})

#Endpoint called when the episode starts
@app.route('/init', methods=['GET'])
def init():
    config = {
        "startMode": "start",  # Options: "start" or "pause"
        "blStartX": 60,  #Blue Start Position
        "blStartY": 10,
        "blStartZ": 27.23,
        "rdStartX": 59, #Red Start Position
        "rdStartY": 10,
        "rdStartZ": 280,
        "trackingMode": False,
        "detectMode": False,
        "logMode": False,
        "stereoCameraMode": False,
        "enemyTracking": False,
        "saveSnapshot": False,
        "saveLog": False,
        "saveLidarData": False,
        "lux": 30000,
        "destoryObstaclesOnHit" : True
    }
    tskijun.init()
    tsinjee.init()
    print("🛠️ Initialization config sent via /init:", config)
    return ts.jsonify(config)

@app.route('/start', methods=['GET'])
def start():
    tskijun.start()
    tsinjee.start()
    print("🚀 /start command received")
    return ts.jsonify({"control": ""})

if __name__ == '__main__':
    app.run(
        host="0.0.0.0",
        port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.39:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Aug/2026 17:55:58] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:55:58] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:55:58] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:55:58] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:55:59] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:55:59] "POST /detect HTTP/1.1" 200 -


Detection count: 1
Tank1 0.953


127.0.0.1 - - [19/Aug/2026 17:55:59] "POST /stereo_image HTTP/1.1" 200 -


탐지된 오브젝트 개수 : 0
[]


...

Detection count: 1
Tank1 0.954


127.0.0.1 - - [19/Aug/2026 17:56:08] "POST /info HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:56:08] "GET /start HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 17:56:08] "POST /stereo_image HTTP/1.1" 200 -


🚀 /start command received
탐지된 오브젝트 개수 : 1
[(86.93475081290792, 11.03692535222634, 65.26312977634146, 'Tank1')]


127.0.0.1 - - [19/Aug/2026 17:56:09] "GET /start HTTP/1.1" 200 -


🚀 /start command received
